In [ ]:
import csv, glob, os, sys, time
from pathlib import Path
T0 = time.time()
COMP = Path(sorted(glob.glob('/kaggle/input/**/test.csv', recursive=True), key=len)[0]).parent
MODELS = Path(sorted(glob.glob('/kaggle/input/**/models.json', recursive=True), key=len)[0]).parent
CODE = Path(sorted(glob.glob(str(MODELS) + '/**/infer_lib.py', recursive=True), key=len)[0]).parent
sys.path.insert(0, str(CODE))
import infer_lib
print('COMP', COMP, 'MODELS', MODELS, 'CODE', CODE, flush=True)


In [ ]:
# scratch train root: gold + reference studies only
import pandas as pd
tr = pd.read_csv(COMP / 'train.csv', dtype={'StudyInstanceUID': str})
LABELS = [c for c in tr.columns if c not in ('StudyInstanceUID', 'Report')]
gold_ids = tr[tr[LABELS].notna().all(axis=1)]['StudyInstanceUID'].tolist()
hold = sorted(glob.glob('/kaggle/input/**/holdout_ref392_ids.csv', recursive=True))
ref_ids = pd.read_csv(hold[0]).iloc[:, 0].astype(str).tolist()
IDS = sorted(set(gold_ids) | set(ref_ids))
FAKE = Path('/tmp/eval_root'); FAKE.mkdir(parents=True, exist_ok=True)
tr[tr['StudyInstanceUID'].isin(set(IDS))][['StudyInstanceUID']].to_csv(FAKE / 'train.csv', index=False)
ser = pd.read_csv(COMP / 'train_series.csv', dtype=str)
ser[ser['StudyInstanceUID'].isin(set(IDS))].to_csv(FAKE / 'train_series.csv', index=False)
os.makedirs(FAKE / 'train_series', exist_ok=True)
for u in IDS:
    src = COMP / 'train_series' / u
    if src.exists() and not (FAKE / 'train_series' / u).exists():
        os.symlink(src, FAKE / 'train_series' / u)
print('studies', len(IDS), flush=True)
infer_lib.predict(str(FAKE), str(MODELS), '/kaggle/working/dk_ref450.csv', split='train')
print('DK_EVAL_DONE minutes %.1f' % ((time.time() - T0) / 60), flush=True)
